# 06 — Limpieza YouTube Comments

Pipeline de limpieza del corpus de comentarios de YouTube sobre IA.
Input:  `youtube_comments_AI_FULL.csv` — 89.160 comentarios en bruto
Output: `data/processed/youtube_comments_clean.parquet` — 62.989 comentarios limpios

Orden del pipeline:
1. Eliminar nulos en `text`
2. Eliminar comentarios del propio canal (spam promocional)
3. Eliminar duplicados por contenido de texto
4. Limpiar texto (URLs, emojis, caracteres especiales)
5. Filtrar por longitud mínima (≥ 10 palabras sobre texto limpio)
6. Clasificar por topic mediante keywords (multi-label)
7. Verificar idioma
8. Guardar parquet

In [1]:
import pandas as pd
import re

pd.set_option("display.max_colwidth", 80)

df = pd.read_csv("youtube_comments_AI_FULL.csv")
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df["comment_year"] = df["date"].dt.year

print(f"Comentarios antes de limpieza: {len(df):,}")

Comentarios antes de limpieza: 89,160


## Filtramos los datos
Eliminamos comentarios irrecuperables antes de tocar el texto.

In [2]:
antes = len(df)
df = df.dropna(subset=["text"])
print(f"Eliminados por text nulo: {antes - len(df):,}")
print(f"Quedan: {len(df):,}")

Eliminados por text nulo: 0
Quedan: 89,160


In [3]:
antes = len(df)
mask_propio = df["author"].str.lstrip("@").str.lower() == df["channel"].str.lower()
df = df[~mask_propio].copy()
print(f"Eliminados del propio canal: {antes - len(df):,}")
print(f"Quedan: {len(df):,}")

Eliminados del propio canal: 13
Quedan: 89,147


In [4]:
antes = len(df)
# keep="first" conserva la aparición más antigua (menor fecha)
df = df.sort_values("date").drop_duplicates(subset="text", keep="first")
print(f"Eliminados por texto duplicado: {antes - len(df):,}")
print(f"Quedan: {len(df):,}")

Eliminados por texto duplicado: 2,140
Quedan: 87,007


## Limpieza de texto
Creamos `text_clean` sin sobreescribir `text` original.
El original se conserva para mostrarlo en el dashboard y el explorador de comentarios.

In [5]:
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    
    # 1. Eliminar URLs
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    
    # 2. Eliminar emojis
    texto = re.sub(
        r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001FA00-\U0001FA6F]",
        " ", texto
    )
    
    # 3. Normalizar saltos de línea y tabulaciones
    texto = re.sub(r"[\n\r\t]+", " ", texto)
    
    # 4. Eliminar caracteres especiales pero conservar puntuación básica
    # Conservamos: letras, números, espacios, y .,!?'"-
    texto = re.sub(r"[^a-zA-Z0-9 .,!?'\"-]", " ", texto)
    
    # 5. Colapsar espacios múltiples
    texto = re.sub(r" {2,}", " ", texto)
    
    # 6. Strip
    texto = texto.strip()
    
    return texto

# Aplicar y crear columna nueva — NO sobreescribimos text original
df["text_clean"] = df["text"].apply(limpiar_texto)

# Verificación visual
print("ORIGINAL:")
print(df["text"].iloc[0])
print("\nLIMPIO:")
print(df["text_clean"].iloc[0])

ORIGINAL:
Thanks for explaining machine learning models with such precision, can’t believe it was so short... I can now decide and lead my team on best ml model to select!

LIMPIO:
Thanks for explaining machine learning models with such precision, can t believe it was so short... I can now decide and lead my team on best ml model to select!


## Filtro de longitud
Umbral: ≥ 10 palabras sobre `text_clean` (no sobre el original).
Justificación: comentarios más cortos no aportan señal semántica suficiente al modelo.
El EDA mostró que el 28.6% del corpus estaba por debajo de este umbral.

In [6]:
antes = len(df)

# Contamos palabras sobre el texto LIMPIO
df["n_palabras_clean"] = df["text_clean"].apply(lambda x: len(x.split()))

df = df[df["n_palabras_clean"] >= 10].copy()

print(f"Eliminados por longitud < 10 palabras: {antes - len(df):,}")
print(f"Quedan: {len(df):,}")
print(f"\nDistribución longitud tras filtro:")
print(df["n_palabras_clean"].describe().apply(lambda x: f"{x:.1f}"))

Eliminados por longitud < 10 palabras: 24,018
Quedan: 62,989

Distribución longitud tras filtro:
count    62989.0
mean        44.1
std         69.7
min         10.0
25%         15.0
50%         25.0
75%         46.0
max       1888.0
Name: n_palabras_clean, dtype: object


## Clasificación por topic (multi-label)
Clasificación por contenido del texto, no por `query_used`.
Un comentario puede pertenecer a education, employment, ambos (both) o ninguno (general).
Las columnas booleanas `is_education` e `is_employment` permiten análisis por sector
incluyendo o excluyendo los "both" según convenga.

In [7]:
# Keywords por categoría — orden importa: education y employment tienen prioridad sobre general
KEYWORDS_EDUCATION = [
    "school", "student", "teacher", "classroom", "homework", "university",
    "college", "education", "learning", "study", "exam", "academic",
    "curriculum", "lecture", "campus", "degree", "tutor", "cheating",
    "kids", "children", "child", "essay", "assignment", "professor",
    "class", "grade", "plagiar", "ban", "pupils", "kindergarten",
    "high school", "primary", "secondary", "teaching", "learn",
    "literacy", "knowledge", "course", "textbook", "library"
]

KEYWORDS_EMPLOYMENT = [
    "job", "jobs", "work", "worker", "employ", "unemploy", "career",
    "salary", "hire", "fired", "layoff", "automat", "replac", "workforce",
    "occupation", "profession", "labor", "labour", "income", "workplace"
]

In [8]:
# Multi-label: un comentario puede pertenecer a varios topics
df["is_employment"] = df["text_clean"].apply(
    lambda t: any(kw in t.lower() for kw in KEYWORDS_EMPLOYMENT) 
    if isinstance(t, str) else False
)
df["is_education"] = df["text_clean"].apply(
    lambda t: any(kw in t.lower() for kw in KEYWORDS_EDUCATION) 
    if isinstance(t, str) else False
)

# Topic principal para análisis simples (cuando necesites una sola etiqueta)
def topic_principal(row):
    if row["is_employment"] and row["is_education"]:
        return "both"
    elif row["is_employment"]:
        return "employment"
    elif row["is_education"]:
        return "education"
    else:
        return "general"

df["topic"] = df.apply(topic_principal, axis=1)

print("Distribución por topic:")
print(df["topic"].value_counts())
print(df["topic"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

Distribución por topic:
topic
general       36895
education     10677
employment    10439
both           4978
Name: count, dtype: int64
topic
general       58.6%
education     17.0%
employment    16.6%
both           7.9%
Name: proportion, dtype: object


In [9]:
# Muestra de comentarios "both" para verificar que tienen sentido
print("Muestra de comentarios clasificados como 'both':")
muestra = df[df["topic"] == "both"][["text_clean", "topic"]].sample(10, random_state=42)
for _, row in muestra.iterrows():
    print(f"\n→ {row['text_clean'][:150]}")

Muestra de comentarios clasificados como 'both':

→ The distinction between 'Prompting' and 'Iterating' is the single most valuable insight for anyone struggling with AI. Most people treat ChatGPT like 

→ All I know is in about 30-40 years office buildings and commuting to work will be pretty much dead. Why physically go into work when you can virtually

→ I haven't watched yet, but the broad term of AI makes it harder to form an opinion. Because AI as a tool vs AI doing your work is very different to me

→ Thanks for the conversation! Remember AI evaluates BASED ON THE INFORMATION WE humans! GIVE AI TO WORK WITH! IF WE GIVE AI the PRIMARY DIRECTIVE to PR

→ These guys are the reason why we as human species will fail. They are totally indoctrinated. It's like listening to some priests and aristocrats 250 y

→ Every day, there is constant research and developments being done by the AI language model companies like OpenAI with stress on limiting misinformatio

→ I have learned to type o

## Verificación de idioma
Estimación sobre muestra de 2.000 comentarios.
Umbral de acción: >5% no inglés → filtro completo. Resultado: 0.2% → sin acción necesaria.

In [10]:
from langdetect import detect, LangDetectException

def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except LangDetectException:
        return "unknown"

# Muestra aleatoria para estimar — no sobre todo el corpus, es lento
muestra = df.sample(2000, random_state=42)
muestra["idioma"] = muestra["text_clean"].apply(detectar_idioma)

print("Distribución de idiomas (muestra 2000 comentarios):")
print(muestra["idioma"].value_counts().head(10))
pct_no_en = (muestra["idioma"] != "en").mean() * 100
print(f"\nEstimación comentarios no inglés en corpus completo: {pct_no_en:.1f}%")

Distribución de idiomas (muestra 2000 comentarios):
idioma
en    1994
id       2
sw       1
nl       1
et       1
it       1
Name: count, dtype: int64

Estimación comentarios no inglés en corpus completo: 0.3%


## Guardado

In [11]:
columnas_output = [
    "comment_id", "video_id", "text", "text_clean",
    "likes", "date", "comment_year", "author",
    "reply_count", "channel", "published_at", "query_used",
    "video_viewCount", "video_commentCount", "video_duration_sec",
    "n_palabras_clean", "topic", "is_education", "is_employment", "period"
]

df[columnas_output].to_parquet(
    "../data/processed/youtube_comments_clean.parquet",
    index=False
)

print(f"Guardado: {len(df):,} comentarios")
print(f"Columnas: {list(df[columnas_output].columns)}")

Guardado: 62,989 comentarios
Columnas: ['comment_id', 'video_id', 'text', 'text_clean', 'likes', 'date', 'comment_year', 'author', 'reply_count', 'channel', 'published_at', 'query_used', 'video_viewCount', 'video_commentCount', 'video_duration_sec', 'n_palabras_clean', 'topic', 'is_education', 'is_employment', 'period']
